# AI Interview Coach

**Steps to be done**
1. Reads a candidate's resume PDF.
2. Splits the resume into small text sections.
3. Converts sections into embeddings and stores them in FAISS.
4. Retrieves relevant resume details (RAG).
5. Generates Technical or HR interview questions using Groq.
6. Accepts typed answers or optional spoken answers (local speech-to-text).
7. Evaluates answers and gives feedback.
8. Calculates an overall interview score.
9. Evaluates RAG/interview-system quality using a model-based test set.


## 1. Install the required libraries

In [1]:
!pip install -q groq pypdf sentence-transformers faiss-cpu

## 2. Import libraries and enter your Groq API key

In [2]:
import json
import re
import getpass
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq

GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "openai/gpt-oss-20b"
print("Libraries and Groq client are ready.")

C:\Users\jiffy\anaconda3\envs\Interviewcoach\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Enter your Groq API key:  ········


Libraries and Groq client are ready.


## 3. Upload and read the resume PDF


In [3]:
!pip install PyPDF2

In [4]:
import os
from PyPDF2 import PdfReader

resume_path = r"C:\Users\jiffy\OneDrive\Documents\AI_Interview_cloud\resume\jiffy11.pdf"

print("Reading PDF...")

reader = PdfReader(resume_path)

print("PDF opened!")

resume_text = ""

for page in reader.pages:
    resume_text += page.extract_text() or ""

print("Extraction completed!")
print("Characters:", len(resume_text))
print(resume_text[:1000])

Reading PDF...
PDF opened!
Extraction completed!
Characters: 2032
JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 Built sales prediction models on Olist e-commerce data using preprocessing, EDA, feature
engineering, and five

## 4. Clean the resume text
Remove email addresses, phone numbers and web links before sending resume content to the language model.

In [5]:
def clean_text(text):
    text = re.sub(r"[\\w.+-]+@[\\w-]+(?:\\.[\\w-]+)+", "", text)
    text = re.sub(r"https?://\\S+|www\\.\\S+", "", text)
    text = re.sub(r"\\b\\d{10}\\b", "", text)
    return re.sub(r"\\s+", " ", text).strip()

resume_text = clean_text(resume_text)
print(resume_text[:1500])

JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 Built sales prediction models on Olist e-commerce data using preprocessing, EDA, feature
engineering, and five ML algorithms.
 Random Forest achieved R2 0.823 and MAE 7,294; a

## 5. Split the resume into chunks
Chunks are small pieces of text. Retrieval searches these pieces to find relevant resume information.

In [6]:
def make_chunks(text, chunk_size=500):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

resume_chunks = make_chunks(resume_text)
print("Number of chunks:", len(resume_chunks))
print(resume_chunks[0][:500])

Number of chunks: 5
JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: Tensor


## 6. Create embeddings and a FAISS index
The MiniLM model converts each chunk into a numerical vector. FAISS helps search for the most relevant chunks.

In [7]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

vectors = embedder.encode(
    resume_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("FAISS index created. Number of vectors:", index.ntotal)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5006.30it/s]


FAISS index created. Number of vectors: 5


## 7. Retrieve relevant resume information
This is the retrieval step in Retrieval-Augmented Generation (RAG).

In [8]:
def retrieve_context(query, top_k=3):
    query_vector = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(query_vector, min(top_k, len(resume_chunks)))
    return "\\n".join(resume_chunks[i] for i in ids[0] if i >= 0)

print(retrieve_context("Tell me about the candidate's machine learning projects"))

JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: Tensor\nFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 Built sales prediction models on Olist e-commerce data using preprocessing, EDA, feature
engineering, and five ML algorithms.
 Random Forest achieved R2 0.823 and MAE 7,294;

## 8. Create a simple function to call Groq
All prompts are sent through this one function.

In [9]:
def ask_groq(prompt, temperature=0.3):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

print(ask_groq("Reply with only the word READY.", temperature=0))

READY


## 9. Generate interview questions
Choose Technical or HR. Technical questions can refer to the resume; HR questions focus on behaviour and motivation.

In [10]:
def generate_questions(job_role, interview_type, number=5):

    context = retrieve_context(job_role + " " + interview_type)

    if interview_type == "Technical":
        instruction = """
        Ask questions about the candidate's resume projects,
        skills and relevant technical concepts.
        """
    else:
        instruction = """
        Ask HR and behavioural questions about motivation,
        teamwork, strengths and challenges.
        Do not ask technical questions.
        """

    prompt = f"""
You are a friendly interviewer for a fresher applying for {job_role}.

Interview round: {interview_type}

Resume information:
{context}

{instruction}

Generate exactly {number} short, clear interview questions.

Return only the questions, one per line, numbered 1 to {number}.
Do not include introductions or explanations.
Do not invent candidate projects or skills.
"""

    reply = ask_groq(prompt, temperature=0.7)

    print("\nDEBUG - Raw Groq response:")
    print(reply)

    if not reply or not isinstance(reply, str):
        print("Error: Groq did not return a valid text response.")
        return []

    questions = re.findall(
        r"^\s*\d+[.)-]\s*(.+?)\s*$",
        reply,
        flags=re.M
    )

    # Fallback if the model returns questions without numbering
    if not questions:
        questions = [
            line.strip("- •").strip()
            for line in reply.splitlines()
            if line.strip() and "?" in line
        ]

    questions = [q for q in questions if q]

    return questions[:number]

## 10. Evaluate an interview answer
The model gives marks for correctness, understanding, completeness and clarity. The Python code calculates the total out of 10.

In [13]:
def evaluate_answer(question, answer, interview_type, job_role):

    context = retrieve_context(question + " " + answer)

    prompt = f"""
You are a fair and supportive interviewer evaluating a fresher.

Job Role: {job_role}
Interview Round: {interview_type}

Question: {question}

Candidate Answer: {answer}

Relevant Resume Information:
{context}

Evaluate the candidate's answer.

Scoring:
1. Correctness: 0 to 4
2. Understanding: 0 to 3
3. Completeness: 0 to 2
4. Clarity: 0 to 1

Provide:
- Strengths
- Missing points
- Simple feedback
- An improved answer

IMPORTANT:
Return ONLY a valid JSON object.
Do not include markdown.
Do not include explanations outside JSON.
Use double quotes for all JSON keys and strings.
Use numbers for marks.
Use arrays for strengths and missing_points.

Required JSON format:

{{
    "correctness": 0,
    "understanding": 0,
    "completeness": 0,
    "clarity": 0,
    "strengths": [],
    "missing_points": [],
    "feedback": "",
    "improved_answer": ""
}}
"""

    result_text = ask_groq(prompt, temperature=0.1)

    # Display response if JSON parsing fails
    try:

        # Remove markdown formatting if present
        result_text = result_text.strip()

        if result_text.startswith("```"):
            result_text = re.sub(
                r"^```(?:json)?\s*|\s*```$",
                "",
                result_text,
                flags=re.I
            )

        # Extract JSON safely
        decoder = json.JSONDecoder()

        start = result_text.find("{")

        if start == -1:
            raise ValueError("No JSON object found.")

        result, end = decoder.raw_decode(result_text[start:])

        # Validate scoring
        limits = {
            "correctness": 4,
            "understanding": 3,
            "completeness": 2,
            "clarity": 1
        }

        for key, maximum in limits.items():

            value = float(result.get(key, 0))

            result[key] = max(0, min(maximum, value))

        # Ensure required fields exist
        result.setdefault("strengths", [])
        result.setdefault("missing_points", [])
        result.setdefault("feedback", "")
        result.setdefault("improved_answer", "")

        # Calculate total score
        result["score"] = sum(
            result[key] for key in limits
        )

        return result

    except Exception as e:

        print("\nJSON PARSING ERROR:", str(e))
        print("\nRAW MODEL RESPONSE:")
        print(result_text)

        raise ValueError(
            "GPT-OSS 20B did not return the expected JSON. "
            "Check the raw response above."
        )

In [16]:
sample_question = "What is Random Forest?"

sample_answer = """
Random Forest is an ensemble learning method
that combines multiple decision trees to improve
prediction accuracy and reduce overfitting.
"""

sample_result = evaluate_answer(
    sample_question,
    sample_answer,
    "Technical",
    Job_Role
)

print("Score:", sample_result["score"], "/ 10")
print("Feedback:", sample_result["feedback"])
print("Strengths:", sample_result["strengths"])
print("Missing points:", sample_result["missing_points"])
print("Improved answer:", sample_result["improved_answer"])

NameError: name 'Job_Role' is not defined

## 11. Run a short mock interview
Type `exit` to stop. The interview asks the generated questions, evaluates each answer and stores the marks.

In [17]:
def run_interview(job_role, interview_type, number_of_questions=5):
    questions = generate_questions(job_role, interview_type, number_of_questions)

    if not questions:
        print("No questions were generated. Check the raw Groq response and ask_groq() function.")
        return

    scores = []

    for number, question in enumerate(questions, 1):
        print(f"\nQuestion {number}: {question}")
        answer = input("Your answer (or type exit): ").strip()

        if answer.lower() == "exit":
            break
        if not answer:
            print("Please enter an answer.")
            continue

        result = evaluate_answer(question, answer, interview_type, job_role)
        scores.append(result["score"])

        print(f"\nMarks: {result['score']} / 10")
        print("Feedback:", result["feedback"])
        print("Strengths:", result["strengths"])
        print("Areas to improve:", result["missing_points"])
        print("Improved answer:", result["improved_answer"])

    if scores:
        average = sum(scores) / len(scores)
        print(f"\nOverall interview score: {average:.1f} / 10")
        print(f"Score out of 5: {average / 2:.1f} / 5")
    else:
        print("No answers were evaluated.")



In [18]:
 run_interview("AI/ML Intern", "Technical", number_of_questions=3)


DEBUG - Raw Groq response:
1. How did you perform feature engineering for the Random Forest model in your e‑commerce sales prediction project?  
2. What evaluation metrics did you use for your recommendation system and how did you handle cold‑start users?  
3. How did you implement TF‑IDF and sentiment classification in your NLP project?

Question 1: How did you perform feature engineering for the Random Forest model in your e‑commerce sales prediction project?


Your answer (or type exit):  I did grouping similar features



Marks: 3.0 / 10
Feedback: Your answer is too brief and lacks the depth expected for a technical interview. Explain the specific preprocessing steps you applied, the rationale behind each, and how they improved the Random Forest model. Include concrete examples and mention any tools or libraries you used.
Strengths: ['Mentioned grouping similar features, indicating some awareness of feature grouping.']
Areas to improve: ['No explanation of how categorical variables were encoded (e.g., one‑hot, target encoding).', 'No discussion of scaling or normalization for numeric features.', 'Lack of detail on creating interaction or polynomial features.', 'No mention of domain‑specific feature creation (e.g., price tiers, time‑based features).', 'Missing use of feature importance from Random Forest for feature selection.', 'No reference to handling missing values or outliers.', 'No clarity on the process or tools used.']
Improved answer: In the e‑commerce sales prediction project, I performed feat

Your answer (or type exit):  exit



Overall interview score: 3.0 / 10
Score out of 5: 1.5 / 5


## 12. Voice-based interview answers

This section lets us answer interview questions by speaking into our laptop microphone.

- Audio is recorded on the computer.
- `faster-whisper` transcribes the recording locally (the audio is not sent to Groq for transcription).
- The transcript is shown so that we can correct it before evaluation.


In [19]:
# Run this cell once to install the optional voice-answer packages.
%pip install -q faster-whisper sounddevice scipy

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import tempfile
import numpy as np
import sounddevice as sd
from scipy.io.wavfile import write
from faster_whisper import WhisperModel

whisper_model = None

def record_and_transcribe_voice(record_seconds=30, sample_rate=16000):
    global whisper_model

    print(f"Recording for up to {record_seconds} seconds. Speak clearly...")
    audio = sd.rec(
        int(record_seconds * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="int16"
    )
    sd.wait()
    print("Recording finished. Transcribing locally...")

    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_file:
            temp_path = temp_file.name

        write(temp_path, sample_rate, audio)

        if whisper_model is None:
            print("Loading Whisper base model (first use may take a little time)...")
            whisper_model = WhisperModel("base", device="cpu", compute_type="int8")

        segments, info = whisper_model.transcribe(temp_path, beam_size=5)
        transcript = " ".join(segment.text.strip() for segment in segments).strip()

        if not transcript:
            print("No speech was detected. Please try recording again.")
            return ""

        print("\nYour spoken answer was transcribed as:")
        print(transcript)
        return transcript

    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)


def get_voice_answer(record_seconds=30):
    transcript = record_and_transcribe_voice(record_seconds)

    if not transcript:
        return ""

    correction = input(
        "\nPress Enter to use this transcript, or type a corrected answer: "
    ).strip()

    return correction if correction else transcript


In [21]:
def run_voice_interview(job_role, interview_type, number_of_questions=5, record_seconds=30):
    questions = generate_questions(job_role, interview_type, number_of_questions)

    if not questions:
        print("No questions were generated. Check the raw Groq response and ask_groq() function.")
        return

    scores = []

    for number, question in enumerate(questions, 1):
        print(f"\nQuestion {number}: {question}")
        choice = input("Press Enter to record a voice answer, or type 'skip'/'exit': ").strip().lower()

        if choice == "exit":
            break
        if choice == "skip":
            continue

        try:
            answer = get_voice_answer(record_seconds)
        except Exception as error:
            print("\nVoice recording/transcription failed:", error)
            print("You can still answer by typing.")
            answer = input("Type your answer (or type exit): ").strip()

        if answer.lower() == "exit":
            break
        if not answer:
            print("No answer was received. Moving to the next question.")
            continue

        result = evaluate_answer(question, answer, interview_type, job_role)
        scores.append(result["score"])

        print(f"\nMarks: {result['score']} / 10")
        print("Feedback:", result["feedback"])
        print("Strengths:", result["strengths"])
        print("Areas to improve:", result["missing_points"])
        print("Improved answer:", result["improved_answer"])

    if scores:
        average = sum(scores) / len(scores)
        print(f"\nOverall interview score: {average:.1f} / 10")
        print(f"Score out of 5: {average / 2:.1f} / 5")
    else:
        print("No answers were evaluated.")


In [22]:
run_voice_interview("AI/ML Intern", "Technical", number_of_questions=3, record_seconds=30)



DEBUG - Raw Groq response:
1. In your ML-Powered E-Commerce Sales Prediction project, what specific feature engineering techniques did you apply before training the Random Forest model, and how did they impact the R² score?  
2. For the NLP text classification project, how did you implement TF‑IDF and which classifier achieved the highest F1‑score?  
3. In your recommendation system work, can you explain the difference between User‑Based and Item‑Based Collaborative Filtering and how you computed cosine similarity in each case?

Question 1: In your ML-Powered E-Commerce Sales Prediction project, what specific feature engineering techniques did you apply before training the Random Forest model, and how did they impact the R² score?


Press Enter to record a voice answer, or type 'skip'/'exit':  


Recording for up to 30 seconds. Speak clearly...
Recording finished. Transcribing locally...
Loading Whisper base model (first use may take a little time)...


C:\Users\jiffy\anaconda3\envs\Interviewcoach\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jiffy\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\jiffy\anaconda3\envs\Interviewcoach\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` c


Your spoken answer was transcribed as:
My did speed feature engineering like that dark leaning and that debris processing steps.



Press Enter to use this transcript, or type a corrected answer:  



Marks: 0 / 10
Feedback: The response lacks any concrete details about the feature engineering process and its impact on the Random Forest model. It does not address the question and does not demonstrate understanding of the project.
Strengths: []
Areas to improve: ['Specific feature engineering techniques applied (e.g., handling missing values, encoding categorical variables, creating interaction terms, scaling, feature selection)', 'Explanation of how each technique improved model performance', 'Quantitative impact on R² score (e.g., R² improved from X to 0.823)', 'Contextual details such as dataset characteristics and preprocessing steps']
Improved answer: In the e‑commerce sales prediction project I performed several targeted feature engineering steps before training the Random Forest:
1. **Missing value imputation** – Numerical columns were filled with median values and categorical columns with the mode, reducing data loss.
2. **One‑hot encoding of categorical variables** – Produc

Press Enter to record a voice answer, or type 'skip'/'exit':  exit



Overall interview score: 0.0 / 10
Score out of 5: 0.0 / 5


## 13. Evaluate the interview model (LLM-as-a-judge)

This section checks the quality of the RAG interview system using test questions. It asks the same Groq model to rate:

1. **Context relevance** – whether retrieved resume context is useful for the test question.
2. **Question relevance** – whether the generated interview question fits the job role and round.
3. **Answer grounding** – whether the system's feedback is supported by the retrieved context and the question.
4. **Evaluation usefulness** – whether the score and feedback are reasonable for the candidate answer.

Each item is scored from 1 to 5. This is an automated, model-based assessment, not a verified ground-truth accuracy score. For a stronger project evaluation, prepare several test cases and manually review a sample of the results.

In [23]:
def _extract_json_object(text):
    text = (text or "").strip()
    # Remove optional Markdown code fences.
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I)
    start = text.find("{")
    if start == -1:
        raise ValueError("The model did not return a JSON object. Raw response:\n" + text)
    decoder = json.JSONDecoder()
    obj, _ = decoder.raw_decode(text[start:])
    return obj


def evaluate_rag_interview_model(test_cases, job_role):
    """
    test_cases is a list of dictionaries with:
      question: interview question to test
      candidate_answer: sample answer
      interview_type: 'Technical' or 'HR'
    """
    evaluation_rows = []

    for case_number, case in enumerate(test_cases, 1):
        question = case["question"]
        candidate_answer = case["candidate_answer"]
        interview_type = case.get("interview_type", "Technical")

        retrieved_context = retrieve_context(question)
        generated_evaluation = evaluate_answer(
            question, candidate_answer, interview_type, job_role
        )

        judge_prompt = f'''
You are an independent evaluator assessing an AI interview-coaching system.

Job role: {job_role}
Interview round: {interview_type}
Test question: {question}
Candidate answer: {candidate_answer}

Retrieved context:
{retrieved_context}

Evaluation produced by the interview system:
{json.dumps(generated_evaluation, ensure_ascii=False)}

Rate each dimension from 1 (poor) to 5 (excellent):
- context_relevance: Is the retrieved context relevant to the question?
- answer_grounding: Is the system feedback consistent with the question, answer and available context? Do not require resume context for general HR questions.
- evaluation_usefulness: Is the score and feedback useful, fair and specific?
- feedback_relevance: Do the strengths, missing points and improved answer address the candidate's actual answer?

Return ONLY one valid JSON object:
{{
  "context_relevance": 1,
  "answer_grounding": 1,
  "evaluation_usefulness": 1,
  "feedback_relevance": 1,
  "reason": "brief explanation"
}}
'''
        judge_text = ask_groq(judge_prompt, temperature=0)
        judge_result = _extract_json_object(judge_text)

        score_keys = [
            "context_relevance",
            "answer_grounding",
            "evaluation_usefulness",
            "feedback_relevance"
        ]
        for key in score_keys:
            judge_result[key] = max(1, min(5, float(judge_result.get(key, 1))))

        judge_result["case_number"] = case_number
        judge_result["question"] = question
        judge_result["interview_score"] = generated_evaluation["score"]
        evaluation_rows.append(judge_result)

        print(f"\nTest case {case_number}: {question}")
        print("Interview-system score:", generated_evaluation["score"], "/ 10")
        print("Model evaluation:", judge_result)

    if evaluation_rows:
        dimensions = [
            "context_relevance",
            "answer_grounding",
            "evaluation_usefulness",
            "feedback_relevance"
        ]
        print("\n========== MODEL EVALUATION SUMMARY ==========")
        for dimension in dimensions:
            average = sum(row[dimension] for row in evaluation_rows) / len(evaluation_rows)
            print(f"{dimension.replace('_', ' ').title()}: {average:.2f} / 5")

        all_scores = [
            row[key]
            for row in evaluation_rows
            for key in dimensions
        ]
        overall = sum(all_scores) / len(all_scores)
        print(f"Overall model-quality rating: {overall:.2f} / 5")

    return evaluation_rows


# Example test set. Add more test cases for a more reliable assessment.
model_test_cases = [
    {
        "question": "Explain the Random Forest algorithm and why it can reduce overfitting.",
        "candidate_answer": "Random Forest combines predictions from multiple decision trees. Each tree is trained using a sample of the data and a subset of features, and combining their predictions can reduce variance.",
        "interview_type": "Technical"
    },
    {
        "question": "What was your role in developing your book recommendation project?",
        "candidate_answer": "I worked on collaborative filtering and used cosine similarity to identify books with similar user preference patterns. I evaluated recommendations using HitRate at 10.",
        "interview_type": "Technical"
    },
    {
        "question": "How do you handle a disagreement with a teammate?",
        "candidate_answer": "I listen to the other person's point of view, explain my reasoning calmly, and try to agree on a solution based on the project goal.",
        "interview_type": "HR"
    }
]


In [24]:
model_evaluation_results = evaluate_rag_interview_model(model_test_cases, "AI/ML Intern")



Test case 1: Explain the Random Forest algorithm and why it can reduce overfitting.
Interview-system score: 7.0 / 10
Model evaluation: {'context_relevance': 1, 'answer_grounding': 5, 'evaluation_usefulness': 5, 'feedback_relevance': 5, 'reason': 'The retrieved resume context is unrelated to the Random Forest question, so context relevance is low. The evaluation feedback accurately reflects the candidate’s answer, is consistent with the question, and provides specific, actionable points, making it well grounded, useful, and relevant.', 'case_number': 1, 'question': 'Explain the Random Forest algorithm and why it can reduce overfitting.', 'interview_score': 7.0}

Test case 2: What was your role in developing your book recommendation project?
Interview-system score: 8.0 / 10
Model evaluation: {'context_relevance': 5, 'answer_grounding': 5, 'evaluation_usefulness': 4.0, 'feedback_relevance': 5, 'reason': "The context directly pertains to the book recommendation project, the feedback accur

**Model Evaluation- Conclusion**

The evaluation of the Personalized AI Interview Coach demonstrates that the system performs well in generating relevant interview feedback and assessing candidate answers. It achieved an overall model-quality rating of **4.00 out of 5**, with particularly good results in answer grounding (4.67), feedback relevance (4.67), and evaluation usefulness (4.33). However, context relevance scored 2.33, indicating that the retrieval component requires further improvement to consistently retrieve the most relevant information from the resume and knowledge base. Overall, the evaluation shows that the system provides a useful foundation for personalized interview preparation. Future improvements will focus on optimizing information retrieval and enhancing the consistency and accuracy of the evaluation process.